# Imports

In [ ]:
# Built-In
from importlib import reload

# 3rd party
import pandas
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tensorflow import keras
from keras import layers
from keras.models import Model, Sequential

# from keras.wrappers.scikit_learn import KerasClassifier
from scikeras.wrappers import KerasClassifier

# In-House


## Intra
Separated because we need to reload often

In [ ]:
import helpers
from helpers import get_damage_profile_composition

reload(helpers)

# Globals

In [ ]:
RANDOM_STATE = 30

# Load Data

In [ ]:
file_path: str = "./merged9.csv"
# Load csv
print(f'Loading "{file_path}"')
data: pandas.DataFrame = pandas.read_csv(file_path)
print(f"Finish loading.")
print(data.shape)
data.head()

In [ ]:
data.columns

# Feature Engineer

## add "Team's Damage Composition"

In [ ]:
data[["blue_physical_damage_perc",
      "blue_magic_damage_perc",
      "blue_true_damage_perc",
      "red_physical_damage_perc",
      "red_magic_damage_perc",
      "red_true_damage_perc"]] = data.apply(get_damage_profile_composition, axis=1, result_type="expand")

# Preprocess

In [ ]:
cols_to_use = [
    "Side",
    "tournament_curr_win_percentage",
    "teams_region",
    "teammate_role_top",
    "teammate_role_jungle",
    "teammate_role_mid",
    "teammate_role_adc",
    "teammate_role_support",
    "enemy_role_top",
    "enemy_role_jungle",
    "enemy_role_mid",
    "enemy_role_adc",
    "enemy_role_support",
    "tournament_type",
    "blue_physical_damage_perc",
    "blue_magic_damage_perc",
    "blue_true_damage_perc",
    "red_physical_damage_perc",
    "red_magic_damage_perc",
    "red_true_damage_perc",
]

X = data[cols_to_use]

## transform target column to discrete values

In [ ]:
y = data["W/L"]
def transform_game_result(game_result):
    if game_result.lower() == "win":
        return 1
    elif game_result.lower() == "loss":
        return 0
    else:
        raise ValueError("\"W/L\" column has invalid values")
y = y.transform(transform_game_result)

## create training, dev, and test set

In [ ]:
# Train Set is 60% of entire data set
# Dev Set is 20% of entire data set
# Test Set is 20% of entire data set
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=RANDOM_STATE)

X_train, X_dev, y_train, y_dev = train_test_split(X_train, y_train, train_size=0.75, test_size=0.25, random_state=RANDOM_STATE)

## get categorical cols

In [ ]:
categorical_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]
print(f"categorical_cols = {categorical_cols}\n")

## get numerical cols

In [ ]:
# Get numerical cols
numerical_cols = [col for col in X_train.columns
                  if X_train[col].dtype in ["int64", "float64"]]
print(f"numerical_cols = {numerical_cols}")

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Build NN

In [ ]:
# def get_model(hidden_layer_dim, meta):
#     n_features_in_ = meta["n_features_in_"]
#     X_shape_ = meta["X_shape_"]
#     n_classes_ = meta["n_classes_"]
#     print(f"X_shape={X_shape_}")
#     model = keras.Sequential(
#         [
#             # layers.Dense(512, activation="relu", input_shape=[X_shape_[1:]]),
#             layers.Dropout(rate=0.5),
#             layers.BatchNormalization(),
#             layers.Dense(512, activation="relu"),
#             layers.Dropout(rate=0.5),
#             layers.BatchNormalization(),
#             # layers.Dense(512, activation='relu'),
#             # layers.Dropout(rate=0.1),
#             # layers.BatchNormalization(),
#             # layers.Dense(512, activation='relu'),
#             # layers.Dropout(rate=0.1),
#             # layers.BatchNormalization(),
#             layers.Dense(1, activation="sigmoid"),
#         ]
#     )

#     model.compile(
#         optimizer="adam", loss="binary_crossentropy", metrics=["binary_accuracy"]
#     )

#     return model


In [ ]:
def get_model(hidden_layer_dim, meta):
    # note that meta is a special argument that will be
    # handed a dict containing input metadata
    n_features_in_ = meta["n_features_in_"]
    X_shape_ = meta["X_shape_"]
    n_classes_ = meta["n_classes_"]

    model = keras.Sequential(
        [
            layers.Dense(n_features_in_, input_shape=X_shape_[1:]),
            layers.Dropout(rate=0.5),
            layers.BatchNormalization(),
            layers.Dense(512, activation="relu"),
            layers.Dropout(rate=0.5),
            layers.BatchNormalization(),
            # layers.Dense(512, activation='relu'),
            # layers.Dropout(rate=0.1),
            # layers.BatchNormalization(),
            # layers.Dense(512, activation='relu'),
            # layers.Dropout(rate=0.1),
            # layers.BatchNormalization(),
            layers.Dense(1, activation="sigmoid"),
        ]
    )
    return model

In [ ]:
nn = KerasClassifier(get_model, loss="binary_crossentropy", hidden_layer_dim=100)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=30,
    min_delta=0.001,
    restore_best_weights=True,
)

param_grid = {
    'nn__optimizer':['rmsprop','adam','adagrad'],
    'nn__epochs':[4,8],
    # 'nn__dropout':[0.1,0.2],
    'nn__callbacks': [early_stopping]
}

In [ ]:
nn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('nn', nn)
])
nn_pipeline.fit(X_train, y_train)

In [ ]:
# pipe = Pipeline([
#     ('scale', OneHotEncoder(handle_unknown="ignore")),
#     ('nn', nn),
# ])
# pipe.fit(X, y)

In [ ]:
grid = GridSearchCV(nn_pipeline, cv=3, param_grid=param_grid)

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
# summarize results
print("Best: %f using %s" % (grid.best_score_, grid.best_params_))
means = grid.cv_results_['mean_test_score']
stds = grid.cv_results_['std_test_score']
params = grid.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))